# Cuaderno ML 1: Segmentación de ciclos de vida de juegos con K-Means

-**Institución Universitaria Pascual Bravo**
- **Asignatura:** Gestión de Datos Masivos
- **Docente:** Genry Gutiérrez
- **Proyecto:** Steam Game Intelligence
- **Infraestructura:** Databricks Lakehouse, Unity Catalog, Delta Lake y Spark MLlib
- **Fuente:** capa Gold (`game_growth`, `dim_game`, `twitch_steam_relationship`)

---

## Pregunta de negocio

> **¿Qué ciclos de vida típicos tienen los juegos de Steam y qué exposición en Twitch tiene cada uno?**

El análisis descriptivo de Gold mostró una correlación prácticamente nula (−0,0032) entre las horas vistas en Twitch y el crecimiento posterior en Steam. Una correlación global mezcla juegos muy distintos. Este cuaderno primero agrupa los juegos según la **forma de su ciclo de vida** y después compara la exposición en Twitch entre esos grupos.

## Objetivos

1. Construir, a partir de Gold, una tabla con **una fila por juego** que describa su ciclo de vida.
2. Seleccionar y preparar las variables para un modelo basado en distancias (escalas, asimetría, correlación).
3. Entrenar **K-Means** con Spark MLlib y elegir el número de grupos con la silueta y el método del codo.
4. Interpretar los arquetipos con una ficha técnica, curvas típicas y composición por género.
5. Comparar la exposición en Twitch entre arquetipos y guardar el resultado en Gold.

---
## 1. Fundamentos: ¿por qué un modelo no supervisado?

| Criterio | No supervisado (este cuaderno) | Supervisado (Cuaderno ML 2) |
| :--- | :--- | :--- |
| **Variable objetivo** | No existe. Nadie etiquetó a los juegos como "longevo" o "de caída rápida". | Sí existe: si el juego creció o no el mes siguiente. |
| **Misión** | Descubrir estructura y perfiles ocultos. | Predecir la etiqueta en datos futuros. |
| **Pregunta** | *¿Qué tipos de ciclo de vida existen?* | *¿Este juego crecerá el próximo mes?* |

### K-Means

K-Means divide $N$ observaciones en $k$ grupos. Cada observación pertenece al grupo con el **centroide** más cercano. El algoritmo minimiza la suma de distancias cuadradas de cada punto a su centroide (inercia):

$$J = \sum_{j=1}^{k} \sum_{\mathbf{x}_i \in C_j} \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert^2$$

1. Elige $k$ centroides iniciales (Spark usa *k-means||*).
2. Asigna cada punto al centroide más cercano.
3. Recalcula cada centroide como el promedio de sus puntos.
4. Repite 2 y 3 hasta que los centroides dejan de moverse.

### Decisiones de diseño para estos datos

1. **Grano juego, no juego-mes.** Si agrupáramos filas mensuales, un mismo juego quedaría repartido en varios clusters y los grupos no se podrían interpretar. Por eso se resume la historia de cada juego en una sola fila.
2. **Solo variables numéricas en la entrada.** En el ejemplo de clase, categorías One-Hot dominantes ("Otro", "Sin Inf") acapararon centroides completos. Aquí el género, el precio y Twitch se usan **después**, para describir los grupos. Así Twitch no define los grupos, pero sí se puede comparar entre ellos.
3. **Transformación logarítmica.** El número de jugadores tiene una cola muy larga (pocos juegos con cientos de miles y miles con menos de 100). Sin `log1p`, los juegos gigantes formarían su propio cluster.
4. **Estandarización obligatoria.** La distancia euclidiana mezcla variables en unidades distintas (meses, proporciones, logaritmos). `StandardScaler` las lleva a media 0 y desviación 1.

---
## 2. Configuración

Plotly no viene preinstalado en el entorno serverless. La instalación va primero porque `%pip` reinicia el intérprete.

In [0]:
%pip install plotly --quiet

In [0]:
import math
import os
from datetime import datetime, timezone

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import PCA, StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql import Window
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("min_months", "12")
dbutils.widgets.text("min_peak_players", "20")
dbutils.widgets.text("k_min", "3")
dbutils.widgets.text("k_max", "8")
dbutils.widgets.text("k_final", "")
dbutils.widgets.text("min_cluster_share", "0.02")
dbutils.widgets.dropdown("persist_results", "true", ["true", "false"])

catalog = dbutils.widgets.get("catalog").strip()
gold_schema = dbutils.widgets.get("gold_schema").strip()
MIN_MONTHS = int(dbutils.widgets.get("min_months"))
MIN_PEAK = float(dbutils.widgets.get("min_peak_players"))
K_MIN = int(dbutils.widgets.get("k_min"))
K_MAX = int(dbutils.widgets.get("k_max"))
K_FINAL_WIDGET = dbutils.widgets.get("k_final").strip()
MIN_SHARE = float(dbutils.widgets.get("min_cluster_share"))
PERSIST = dbutils.widgets.get("persist_results") == "true"

if K_MIN < 2 or K_MAX < K_MIN:
    raise ValueError("Se requiere 2 <= k_min <= k_max")

SEED = 42
TWITCH_START = "2016-01-01"
TWITCH_END = "2024-09-01"
OUTPUT_TABLE = "game_lifecycle_clusters"


def gold_table(name):
    return f"`{catalog}`.`{gold_schema}`.`{name}`"


# Tablas que se exportan al final en Markdown para revisar los resultados fuera de Databricks.
REPORT = []


def add_report(title, pdf):
    REPORT.append((title, pdf.copy()))
    return pdf


print(f"Gold: {catalog}.{gold_schema} | min_months={MIN_MONTHS} | min_peak_players={MIN_PEAK} | k en [{K_MIN}, {K_MAX}]")

---
## 3. Carga de la capa Gold

In [0]:
game_growth = spark.table(gold_table("game_growth"))
dim_game = spark.table(gold_table("dim_game"))
twitch = spark.table(gold_table("twitch_steam_relationship"))

volumetria = pd.DataFrame([
    {"tabla": "game_growth", "filas": game_growth.count(), "juegos": game_growth.select("app_id").distinct().count()},
    {"tabla": "dim_game", "filas": dim_game.count(), "juegos": dim_game.select("app_id").distinct().count()},
    {"tabla": "twitch_steam_relationship", "filas": twitch.count(), "juegos": twitch.select("app_id").distinct().count()},
])
display(add_report("Volumetría de las tablas Gold", volumetria))
display(game_growth.orderBy("app_id", "month_start").limit(10))

---
## 4. Ingeniería de variables por juego

Se parte de `gold.game_growth` (una fila por juego y mes) y se resume la historia de cada juego en siete variables:

| Variable | Qué mide | Cálculo |
|---|---|---|
| `log_peak_players` | Escala de la comunidad | `log1p(máximo de avg_players)` |
| `months_observed` | Longevidad | Número de meses con medición |
| `pct_time_to_peak` | Momento del pico | (mes del pico − primer mes) ÷ (último mes − primer mes). 0 = pico al inicio, 1 = pico al final |
| `retention_ratio` | Retención | Promedio de los últimos 3 meses ÷ máximo |
| `trend_slope_yearly` | Tendencia | Pendiente de `log1p(avg_players)` contra el tiempo, por año |
| `volatility` | Estabilidad | Desviación estándar del cambio mensual en logaritmo (solo meses consecutivos) |
| `log_peak_to_avg` | Picos dentro del mes | `log(promedio de peak_to_avg_ratio)` |

La volatilidad usa diferencias de logaritmos y no `mom_change_pct`. Ese porcentaje explota cuando el mes anterior tiene casi cero jugadores, una limitación ya documentada en el análisis de Gold.

In [0]:
monthly = (
    game_growth
    .filter(F.col("avg_players").isNotNull())
    .select(
        "app_id",
        "name",
        "month_start",
        F.col("avg_players").cast("double").alias("avg_players"),
        F.col("prev_avg_players").cast("double").alias("prev_avg_players"),
        F.col("peak_to_avg_ratio").cast("double").alias("peak_to_avg_ratio"),
        F.col("mom_change_pct").cast("double").alias("mom_change_pct"),
    )
    .withColumn("month_idx", F.year("month_start") * 12 + F.month("month_start"))
    .withColumn("log_players", F.log1p("avg_players"))
    # mom_change_pct solo existe cuando el mes anterior es consecutivo y tiene jugadores.
    .withColumn(
        "log_diff",
        F.when(F.col("mom_change_pct").isNotNull(), F.log1p("avg_players") - F.log1p("prev_avg_players")),
    )
    .withColumn("rn_desc", F.row_number().over(Window.partitionBy("app_id").orderBy(F.desc("month_start"))))
)

per_game = (
    monthly.groupBy("app_id")
    .agg(
        F.max("name").alias("name"),
        F.count("*").alias("months_observed"),
        F.min("month_start").alias("first_month"),
        F.max("month_start").alias("last_month"),
        F.min("month_idx").alias("first_idx"),
        F.max("month_idx").alias("last_idx"),
        F.max("avg_players").alias("peak_players"),
        # Mes del pico: ante empates se toma el primero (mayor -month_idx).
        F.max(F.struct(F.col("avg_players"), (-F.col("month_idx")).alias("neg_idx"))).alias("peak_struct"),
        F.avg(F.when(F.col("rn_desc") <= 3, F.col("avg_players"))).alias("avg_last3"),
        F.expr("regr_slope(log_players, month_idx)").alias("slope_monthly"),
        F.stddev("log_diff").alias("volatility"),
        F.avg("peak_to_avg_ratio").alias("mean_peak_to_avg"),
    )
    .withColumn("peak_idx", -F.col("peak_struct.neg_idx"))
    .drop("peak_struct")
    .cache()
)

print(f"Juegos con historial en game_growth: {per_game.count():,}")

### 4.1 Población del modelo (Fase 0: validación de umbrales)

Se requieren dos filtros antes de agrupar:

- **Historial mínimo** (`min_months`): sin suficientes meses no se puede hablar de ciclo de vida.
- **Pico mínimo de jugadores** (`min_peak_players`): con bases casi nulas, los cambios relativos son ruido.

La tabla muestra cuántos juegos quedan con distintas combinaciones de umbrales. Sirve para justificar los valores de los widgets.

In [0]:
month_thresholds = [6, 12, 24]
peak_thresholds = [0, 20, 50, 100]
grid_row = per_game.agg(*[
    F.sum(F.when((F.col("months_observed") >= m) & (F.col("peak_players") >= p), 1).otherwise(0)).alias(f"m{m}_p{p}")
    for m in month_thresholds
    for p in peak_thresholds
]).first()
threshold_grid = pd.DataFrame(
    [[grid_row[f"m{m}_p{p}"] for p in peak_thresholds] for m in month_thresholds],
    index=[f"meses >= {m}" for m in month_thresholds],
    columns=[f"pico >= {p}" for p in peak_thresholds],
).reset_index().rename(columns={"index": "historial"})
display(add_report("Juegos que cumplen cada combinación de umbrales", threshold_grid))

per_game_pd = per_game.select("months_observed", "peak_players").toPandas()
per_game_pd["log10_peak_players"] = (per_game_pd["peak_players"] + 1).apply(math.log10)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Meses observados por juego", "Pico de jugadores por juego (log10)"))
fig.add_trace(go.Histogram(x=per_game_pd["months_observed"], nbinsx=60, name="meses"), row=1, col=1)
fig.add_trace(go.Histogram(x=per_game_pd["log10_peak_players"], nbinsx=60, name="log10 pico"), row=1, col=2)
fig.add_vline(x=MIN_MONTHS, line_dash="dash", line_color="red", row=1, col=1)
fig.add_vline(x=math.log10(MIN_PEAK + 1), line_dash="dash", line_color="red", row=1, col=2)
fig.update_layout(template="plotly_white", height=420, showlegend=False, title="Distribución del historial y la escala de los juegos (línea roja = umbral)")
fig.show()

In [0]:
FEATURES = [
    "log_peak_players",
    "months_observed",
    "pct_time_to_peak",
    "retention_ratio",
    "trend_slope_yearly",
    "volatility",
    "log_peak_to_avg",
]

population = per_game.filter((F.col("months_observed") >= MIN_MONTHS) & (F.col("peak_players") >= MIN_PEAK))

games = (
    population
    .withColumn("log_peak_players", F.log1p("peak_players"))
    .withColumn("months_observed", F.col("months_observed").cast("double"))
    .withColumn(
        "pct_time_to_peak",
        (F.col("peak_idx") - F.col("first_idx")) / F.greatest(F.col("last_idx") - F.col("first_idx"), F.lit(1)),
    )
    .withColumn("retention_ratio", F.expr("try_divide(avg_last3, peak_players)"))
    .withColumn("trend_slope_yearly", F.col("slope_monthly") * 12)
    .withColumn("log_peak_to_avg", F.log(F.greatest(F.col("mean_peak_to_avg"), F.lit(1.0))))
)

n_population = games.count()
games_model = games.na.drop(subset=FEATURES).orderBy("app_id").coalesce(1).cache()
n_model = games_model.count()

print(f"Juegos que cumplen los umbrales: {n_population:,}")
print(f"Juegos descartados por variables nulas: {n_population - n_model:,}")
print(f"Juegos que entran al modelo: {n_model:,}")

### 4.2 Selección de variables: estadísticos y correlación

Antes de entrenar se revisa la escala de cada variable y su correlación. Dos variables con correlación muy alta (|r| > 0,85) dan casi la misma información y pesan doble en la distancia. Si aparece un par así, conviene retirar una de las dos.

In [0]:
feature_summary = (
    games_model.select(FEATURES)
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
    .toPandas()
)
display(add_report("Estadísticos de las variables de entrada", feature_summary))

features_pd = games_model.select(FEATURES).toPandas()
correlation = features_pd.corr().round(2)
add_report("Matriz de correlación de las variables de entrada", correlation.reset_index().rename(columns={"index": "variable"}))

fig = px.imshow(
    correlation,
    text_auto=".2f",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    title="Correlación entre las variables de ciclo de vida",
)
fig.update_layout(template="plotly_white", height=560)
fig.show()

high_pairs = [
    (a, b, correlation.loc[a, b])
    for i, a in enumerate(FEATURES)
    for b in FEATURES[i + 1:]
    if abs(correlation.loc[a, b]) > 0.85
]
print("Pares con |r| > 0,85:", high_pairs if high_pairs else "ninguno")

---
## 5. Pipeline de preparación

```
[7 variables numéricas]
        │
        ▼
VectorAssembler   → 'raw_features'
        │
        ▼
StandardScaler (withMean=True, withStd=True) → 'features'
```

A diferencia del ejemplo de clase, aquí se usa `withMean=True`. En clase era obligatorio `withMean=False` porque el One-Hot produce vectores dispersos y centrar los volvería densos. Este vector ya es denso, así que se puede centrar sin costo.

**Reproducibilidad:** en el ejemplo de clase, la misma semilla dio tres particiones distintas en serverless, porque la inicialización depende de cómo se reparten los datos. Aquí los datos se ordenan por `app_id` y se reúnen en una sola partición (`coalesce(1)`) antes de entrenar. Con unos miles de juegos no afecta el tiempo y la ejecución se vuelve repetible.

In [0]:
assembler = VectorAssembler(inputCols=FEATURES, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
prep_model = Pipeline(stages=[assembler, scaler]).fit(games_model)
df_prep = prep_model.transform(games_model).cache()

scaler_model = prep_model.stages[-1]
scaling = pd.DataFrame({
    "variable": FEATURES,
    "media": scaler_model.mean.toArray(),
    "desviacion": scaler_model.std.toArray(),
})
display(scaling)
df_prep.select("app_id", "name", "features").show(3, truncate=100)

---
## 6. ¿Cuántos arquetipos existen? Silueta y método del codo

En el aprendizaje no supervisado no se conoce de antemano el valor de $k$. Se usan tres criterios:

1. **Coeficiente de silueta** (más alto es mejor):
   $$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
   donde $a(i)$ es la distancia media de $i$ a su propio cluster (cohesión) y $b(i)$ la distancia media al cluster vecino más cercano (separación). Varía entre −1 y +1.
2. **Inercia** (método del codo): la suma de distancias cuadradas siempre baja al aumentar $k$. Se busca el punto donde deja de bajar con fuerza.
3. **Tamaño mínimo de cluster:** se descarta un $k$ que deje un grupo con menos del `min_cluster_share` de los juegos. En clase, un cluster de 24 registros resultó ser un artefacto de la inicialización y no un arquetipo.

Si el widget `k_final` está vacío, se elige el $k$ con mayor silueta entre los que cumplen el tamaño mínimo.

In [0]:
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean",
)

k_rows = []
for k in range(K_MIN, K_MAX + 1):
    model_k = KMeans(featuresCol="features", predictionCol="prediction", k=k, seed=SEED, maxIter=100).fit(df_prep)
    pred_k = model_k.transform(df_prep)
    sizes = model_k.summary.clusterSizes
    k_rows.append({
        "k": k,
        "silhouette": round(evaluator.evaluate(pred_k), 4),
        "inercia": round(model_k.summary.trainingCost, 2),
        "min_cluster_share": round(min(sizes) / sum(sizes), 4),
        "tamanos": str(sorted(sizes, reverse=True)),
    })
    print(f"k={k} | silueta={k_rows[-1]['silhouette']:.4f} | inercia={k_rows[-1]['inercia']:,.0f} | cluster más pequeño={k_rows[-1]['min_cluster_share']:.1%}")

k_results = pd.DataFrame(k_rows)
k_results["cumple_tamano_minimo"] = k_results["min_cluster_share"] >= MIN_SHARE

if K_FINAL_WIDGET:
    K_FINAL = int(K_FINAL_WIDGET)
    k_reason = "fijado manualmente con el widget k_final"
else:
    candidates = k_results[k_results["cumple_tamano_minimo"]]
    if candidates.empty:
        candidates = k_results
        print("Ningún k cumple el tamaño mínimo; se elige solo por silueta.")
    K_FINAL = int(candidates.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0]["k"])
    k_reason = "mayor silueta entre los k que cumplen el tamaño mínimo"

display(add_report("Evaluación de k (silueta, inercia y tamaño mínimo)", k_results))
print(f"k elegido: {K_FINAL} ({k_reason})")

In [0]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=k_results["k"], y=k_results["silhouette"], mode="lines+markers", name="Silueta"), secondary_y=False)
fig.add_trace(go.Scatter(x=k_results["k"], y=k_results["inercia"], mode="lines+markers", name="Inercia (codo)", line=dict(dash="dot")), secondary_y=True)
fig.add_vline(x=K_FINAL, line_dash="dash", line_color="red", annotation_text=f"k elegido = {K_FINAL}")
fig.update_layout(template="plotly_white", height=460, title="Selección del número de arquetipos")
fig.update_xaxes(title_text="k (número de clusters)", dtick=1)
fig.update_yaxes(title_text="Silueta (más alto es mejor)", secondary_y=False)
fig.update_yaxes(title_text="Inercia", secondary_y=True)
fig.show()

---
## 7. Modelo final

Los identificadores que entrega K-Means son arbitrarios: pueden cambiar de orden entre ejecuciones aunque los grupos sean los mismos. Para que la interpretación sea estable, se renumeran según la escala del centroide: **el cluster 0 es el de menor pico de jugadores** y el último el de mayor pico.

Después de revisar la ficha técnica (sección 8) se completa `CLUSTER_NAMES` con un nombre de negocio para cada arquetipo y se vuelve a ejecutar desde esta celda.

In [0]:
# Completar después de revisar la ficha técnica, por ejemplo:
# CLUSTER_NAMES = {0: "Nicho pequeño", 1: "Lanzamiento con caída rápida", 2: "Longevo estable"}
CLUSTER_NAMES = {}

kmeans_model = KMeans(featuresCol="features", predictionCol="prediction", k=K_FINAL, seed=SEED, maxIter=100).fit(df_prep)
pred_final = kmeans_model.transform(df_prep)
SILHOUETTE_FINAL = evaluator.evaluate(pred_final)

centers_scaled = pd.DataFrame(kmeans_model.clusterCenters(), columns=FEATURES)
order = centers_scaled["log_peak_players"].sort_values().index.tolist()
old_to_new = {old: new for new, old in enumerate(order)}
relabel = F.element_at(F.array(*[F.lit(old_to_new[i]) for i in range(K_FINAL)]), F.col("prediction") + 1)

clustered = pred_final.withColumn("cluster_id", relabel.cast("int")).drop("prediction").cache()
centers_scaled.index = [old_to_new[i] for i in centers_scaled.index]
centers_scaled = centers_scaled.sort_index()
centers_original = centers_scaled * scaler_model.std.toArray() + scaler_model.mean.toArray()


def cluster_label(cluster_id):
    cluster_id = int(cluster_id)
    return f"C{cluster_id} · {CLUSTER_NAMES[cluster_id]}" if cluster_id in CLUSTER_NAMES else f"Cluster {cluster_id}"


sizes_pd = clustered.groupBy("cluster_id").count().orderBy("cluster_id").toPandas()
sizes_pd["pct_juegos"] = (100 * sizes_pd["count"] / sizes_pd["count"].sum()).round(1)
sizes_pd["arquetipo"] = sizes_pd["cluster_id"].apply(cluster_label)

print(f"K-Means final: k={K_FINAL} | silueta={SILHOUETTE_FINAL:.4f} | juegos={n_model:,}")
display(add_report("Tamaño de cada cluster", sizes_pd))

centers_report = centers_original.round(3).reset_index().rename(columns={"index": "cluster_id"})
display(add_report("Centroides en unidades originales", centers_report))

---
## 8. Perfilamiento e interpretación de los arquetipos

Los números del modelo no le dicen nada a un estudio o a un publisher por sí solos. Aquí se describe cada cluster con sus variables de entrada y con información que **no** se usó para agrupar:

- **Catálogo (`dim_game`):** precio, % de juegos gratuitos, año de lanzamiento, géneros y proporción de reseñas positivas.
- **Twitch (`twitch_steam_relationship`):** si el juego estuvo alguna vez en el top 200 mensual, cuántos meses y cuántas horas vistas. Solo se comparan juegos observados dentro de la ventana con datos de Twitch (2016-01 a 2024-09), para no contar como "sin exposición" a juegos de los que simplemente no hay datos.

In [0]:
twitch_game = twitch.groupBy("app_id").agg(
    F.count("*").alias("months_in_twitch_top200"),
    F.sum("hours_watched").alias("total_hours_watched"),
    F.min("twitch_rank").alias("best_twitch_rank"),
)

profile = (
    clustered.select("app_id", "name", "cluster_id", "peak_players", "peak_idx", "first_month", "last_month", *FEATURES)
    .join(dim_game.select("app_id", "price", "genres", "release_date", "review_ratio"), "app_id", "left")
    .join(twitch_game, "app_id", "left")
    .withColumn("is_free", F.when(F.col("price").isNull(), None).otherwise((F.col("price") == 0).cast("double")))
    .withColumn("release_year", F.year("release_date"))
    .withColumn(
        "in_twitch_window",
        (F.col("last_month") >= F.to_date(F.lit(TWITCH_START))) & (F.col("first_month") <= F.to_date(F.lit(TWITCH_END))),
    )
    .fillna({"months_in_twitch_top200": 0, "total_hours_watched": 0})
    .withColumn("ever_in_twitch_top200", (F.col("months_in_twitch_top200") > 0).cast("double"))
    .cache()
)

ficha = (
    profile.groupBy("cluster_id")
    .agg(
        F.count("*").alias("juegos"),
        F.round(F.percentile_approx("peak_players", 0.5), 0).alias("mediana_pico_jugadores"),
        F.percentile_approx("months_observed", 0.5).alias("mediana_meses_observados"),
        F.round(F.percentile_approx("pct_time_to_peak", 0.5), 2).alias("mediana_pct_tiempo_al_pico"),
        F.round(F.percentile_approx("retention_ratio", 0.5), 3).alias("mediana_retencion"),
        F.round(F.percentile_approx("trend_slope_yearly", 0.5), 3).alias("mediana_tendencia_anual_log"),
        F.round(F.percentile_approx("volatility", 0.5), 3).alias("mediana_volatilidad"),
        F.round(F.exp(F.percentile_approx("log_peak_to_avg", 0.5)), 2).alias("mediana_pico_sobre_promedio"),
        F.round(100 * F.avg("is_free"), 1).alias("pct_gratuitos"),
        F.round(F.avg("price"), 2).alias("precio_medio"),
        F.percentile_approx("release_year", 0.5).alias("mediana_anio_lanzamiento"),
        F.round(F.avg("review_ratio"), 3).alias("review_ratio_medio"),
    )
    .orderBy("cluster_id")
    .toPandas()
)
ficha.insert(1, "arquetipo", ficha["cluster_id"].apply(cluster_label))
ficha.insert(3, "pct_juegos", (100 * ficha["juegos"] / ficha["juegos"].sum()).round(1))
display(add_report("Ficha técnica por arquetipo", ficha))

In [0]:
# Géneros: un juego puede tener varios, así que se calcula el % de juegos del cluster que tiene cada género.
genre_long = profile.select("cluster_id", "app_id", F.explode("genres").alias("genre"))
top_genres = [
    row["genre"]
    for row in genre_long.groupBy("genre").agg(F.countDistinct("app_id").alias("n")).orderBy(F.desc("n"), "genre").limit(8).collect()
]

genre_pct = (
    genre_long.filter(F.col("genre").isin(top_genres))
    .groupBy("cluster_id", "genre").agg(F.countDistinct("app_id").alias("juegos_con_genero"))
    .join(profile.groupBy("cluster_id").agg(F.count("*").alias("juegos")), "cluster_id")
    .withColumn("pct", F.round(100 * F.col("juegos_con_genero") / F.col("juegos"), 1))
    .toPandas()
)
genre_matrix = genre_pct.pivot(index="cluster_id", columns="genre", values="pct").reindex(columns=top_genres).fillna(0).sort_index()
genre_matrix.index = [cluster_label(c) for c in genre_matrix.index]
add_report("Porcentaje de juegos de cada cluster por género (top 8)", genre_matrix.reset_index().rename(columns={"index": "arquetipo"}))

fig = px.imshow(
    genre_matrix,
    text_auto=".1f",
    color_continuous_scale="Blues",
    aspect="auto",
    title="Composición de géneros por arquetipo (% de juegos del cluster con ese género)",
    labels={"x": "Género", "y": "Arquetipo", "color": "% juegos"},
)
fig.update_layout(template="plotly_white", height=480)
fig.show()

In [0]:
# Juegos de ejemplo: los 5 con mayor pico de cada cluster ayudan a ponerle nombre al arquetipo.
window_top = Window.partitionBy("cluster_id").orderBy(F.desc("peak_players"), "app_id")
examples = (
    profile.withColumn("rank_in_cluster", F.row_number().over(window_top))
    .filter(F.col("rank_in_cluster") <= 5)
    .select(
        "cluster_id",
        "rank_in_cluster",
        "app_id",
        "name",
        F.round("peak_players", 0).alias("pico_jugadores"),
        F.col("months_observed").cast("int").alias("meses"),
        F.round("retention_ratio", 3).alias("retencion"),
        "release_year",
    )
    .orderBy("cluster_id", "rank_in_cluster")
    .toPandas()
)
display(add_report("Juegos de ejemplo por cluster (top 5 por pico)", examples))

### 8.1 Curva típica de cada arquetipo

Este es el gráfico central del análisis. Para cada juego se alinea su historia respecto al **mes de su pico** (mes 0) y se expresa el número de jugadores como % del pico. Después se toma la mediana por arquetipo. Así se compara la **forma** del ciclo de vida sin que la escala de cada juego influya.

In [0]:
curve = (
    monthly.select("app_id", "month_idx", "avg_players")
    .join(clustered.select("app_id", "cluster_id", "peak_idx", "peak_players"), "app_id")
    .withColumn("months_from_peak", F.col("month_idx") - F.col("peak_idx"))
    .filter(F.col("months_from_peak").between(-24, 60))
    .withColumn("pct_of_peak", F.expr("try_divide(avg_players, peak_players)") * 100)
    .groupBy("cluster_id", "months_from_peak")
    .agg(
        F.percentile_approx("pct_of_peak", 0.5).alias("mediana_pct_pico"),
        F.countDistinct("app_id").alias("juegos"),
    )
    .filter(F.col("juegos") >= 20)
    .orderBy("cluster_id", "months_from_peak")
    .toPandas()
)
curve["arquetipo"] = curve["cluster_id"].apply(cluster_label)

fig = px.line(
    curve,
    x="months_from_peak",
    y="mediana_pct_pico",
    color="arquetipo",
    title="Curva típica de jugadores por arquetipo (mediana, % del pico)",
    labels={"months_from_peak": "Meses respecto al pico (0 = mes del pico)", "mediana_pct_pico": "% del pico de jugadores", "arquetipo": "Arquetipo"},
    template="plotly_white",
)
fig.add_vline(x=0, line_dash="dot", line_color="gray")
fig.update_layout(height=520)
fig.show()

curve_summary = curve[curve["months_from_peak"].isin([-12, -6, 0, 6, 12, 24, 36, 60])].pivot(
    index="arquetipo", columns="months_from_peak", values="mediana_pct_pico"
).round(1)
add_report("Curva típica: % del pico en meses seleccionados", curve_summary.reset_index())
display(curve_summary.reset_index())

### 8.2 Perfil de los centroides y mapa PCA

- **Mapa de calor de centroides:** cada celda indica cuántas desviaciones estándar se aleja el centroide del promedio general en cada variable. Rojo = por encima del promedio; azul = por debajo. Es la forma más rápida de leer qué distingue a cada arquetipo.
- **PCA 2D:** reduce las 7 variables a 2 componentes principales para ver cómo se separan los grupos. Es una proyección, así que parte de la separación real no se ve en 2 dimensiones.

In [0]:
heat = centers_scaled.copy()
heat.index = [cluster_label(c) for c in heat.index]
fig = px.imshow(
    heat.round(2),
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-2,
    zmax=2,
    aspect="auto",
    title="Centroides estandarizados (desviaciones respecto al promedio)",
    labels={"x": "Variable", "y": "Arquetipo", "color": "z"},
)
fig.update_layout(template="plotly_white", height=460)
fig.show()

pca_model = PCA(k=2, inputCol="features", outputCol="pca_features").fit(clustered)
explained = pca_model.explainedVariance.toArray()
pca_pd = (
    pca_model.transform(clustered)
    .select("app_id", "name", "cluster_id", "peak_players", "months_observed", vector_to_array("pca_features").alias("pc"))
    .toPandas()
)
pca_pd["PC1"] = pca_pd["pc"].apply(lambda v: float(v[0]))
pca_pd["PC2"] = pca_pd["pc"].apply(lambda v: float(v[1]))
pca_pd["arquetipo"] = pca_pd["cluster_id"].apply(cluster_label)

fig = px.scatter(
    pca_pd.sort_values("cluster_id"),
    x="PC1",
    y="PC2",
    color="arquetipo",
    hover_data=["name", "peak_players", "months_observed"],
    opacity=0.6,
    title=f"Mapa 2D de los arquetipos (PCA: {explained[0]:.0%} + {explained[1]:.0%} de la varianza)",
    template="plotly_white",
)
fig.update_layout(height=600, legend_title_text="Arquetipo")
fig.show()

### 8.3 Exposición en Twitch por arquetipo (pregunta de negocio)

Se comparan solo los juegos observados dentro de la ventana 2016-01 a 2024-09. La comparación es **descriptiva**: si un arquetipo concentra más exposición en Twitch, existe una asociación, pero no prueba que Twitch cause ese ciclo de vida. También puede ocurrir lo contrario: que los juegos con comunidades grandes y estables atraigan más streams.

In [0]:
twitch_by_cluster = (
    profile.filter(F.col("in_twitch_window"))
    .groupBy("cluster_id")
    .agg(
        F.count("*").alias("juegos_en_ventana"),
        F.round(100 * F.avg("ever_in_twitch_top200"), 1).alias("pct_alguna_vez_top200"),
        F.round(F.avg("months_in_twitch_top200"), 2).alias("meses_top200_promedio"),
        F.percentile_approx(
            F.when(F.col("ever_in_twitch_top200") == 1, F.col("total_hours_watched")), 0.5
        ).alias("mediana_horas_vistas_si_aparece"),
        F.min("best_twitch_rank").alias("mejor_rank"),
    )
    .orderBy("cluster_id")
    .toPandas()
)
twitch_by_cluster.insert(1, "arquetipo", twitch_by_cluster["cluster_id"].apply(cluster_label))
display(add_report("Exposición en Twitch por arquetipo (juegos dentro de la ventana 2016-01 a 2024-09)", twitch_by_cluster))

fig = make_subplots(rows=1, cols=2, subplot_titles=("% de juegos que alguna vez estuvo en el top 200", "Meses promedio en el top 200"))
fig.add_trace(go.Bar(x=twitch_by_cluster["arquetipo"], y=twitch_by_cluster["pct_alguna_vez_top200"], text=twitch_by_cluster["pct_alguna_vez_top200"], name="% top 200"), row=1, col=1)
fig.add_trace(go.Bar(x=twitch_by_cluster["arquetipo"], y=twitch_by_cluster["meses_top200_promedio"], text=twitch_by_cluster["meses_top200_promedio"], name="meses"), row=1, col=2)
fig.update_layout(template="plotly_white", height=460, showlegend=False, title="Exposición en Twitch por arquetipo de ciclo de vida")
fig.show()

---
## 9. Interpretación de resultados

> **Completar con los resultados de la ejecución.** Usar la ficha técnica, la curva típica, el mapa de calor de centroides, los juegos de ejemplo y la tabla de Twitch. Citar siempre los números obtenidos.

### Calidad de la segmentación
- k elegido y por qué (silueta, codo, tamaño mínimo): _…_
- Silueta final y cómo se interpreta: _…_

### Arquetipos encontrados
- **Cluster 0 — _nombre_ (_x_ % de los juegos):** _forma de la curva, retención, longevidad, géneros, juegos de ejemplo…_
- **Cluster 1 — _nombre_:** _…_
- _…_

### Exposición en Twitch
- ¿Qué arquetipo concentra más presencia en el top 200? _…_
- ¿Qué significa para la pregunta de negocio (asociación, no causalidad)? _…_

### Lo que no separó a los grupos
- _Variables cuyo centroide es parecido en todos los clusters…_

### Aplicaciones en el proyecto
1. **Segmentación del catálogo:** comparar un juego con otros de su mismo arquetipo y no con el promedio de Steam.
2. **Señal de alerta:** un juego cuya curva se parece a la de un arquetipo de caída rápida puede priorizarse para acciones de retención.
3. **Variable para otros modelos:** el arquetipo resume la historia del juego en una sola etiqueta.

### Limitaciones
- Los juegos lanzados antes de 2012 se observan desde la mitad de su vida, así que su curva está incompleta.
- La segmentación usa la historia completa de cada juego. No sirve como variable predictiva sin recalcularla con datos hasta la fecha de la predicción.
- El cruce con Twitch usa solo nombres con coincidencia única; los juegos con nombre ambiguo cuentan como "sin exposición".

---
## 10. Persistencia en Gold

El resultado se guarda en `gold.game_lifecycle_clusters` para usarlo en tableros o en otros análisis. La escritura reemplaza la tabla completa, así que repetir la ejecución no duplica filas.

In [0]:
if PERSIST:
    cluster_names = [CLUSTER_NAMES.get(i, f"Cluster {i}") for i in range(K_FINAL)]
    to_save = (
        profile.select("app_id", "name", "cluster_id", "peak_players", *FEATURES)
        .withColumn("cluster_name", F.element_at(F.array(*[F.lit(n) for n in cluster_names]), F.col("cluster_id") + 1))
        .withColumn("algorithm", F.lit("kmeans"))
        .withColumn("k", F.lit(K_FINAL))
        .withColumn("silhouette", F.lit(float(SILHOUETTE_FINAL)))
        .withColumn("generated_at", F.current_timestamp())
    )
    (
        to_save.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(gold_table(OUTPUT_TABLE))
    )
    print(f"Tabla guardada: {catalog}.{gold_schema}.{OUTPUT_TABLE} ({to_save.count():,} filas)")
else:
    print("persist_results=false: no se escribió ninguna tabla.")

---
## 11. Exportación de resultados

Guarda las tablas principales en `notebooks/data_validations/result_outputs/resultados_ml_01_kmeans.md`, dentro del Git Folder, para revisar los resultados fuera de Databricks.

In [0]:
def markdown_value(value):
    if value is None:
        return ""
    if isinstance(value, float):
        if math.isnan(value):
            return ""
        return f"{value:.6g}"
    return str(value).replace("|", "\\|").replace("\n", " ")


def to_markdown(pdf, max_rows=200):
    pdf = pdf.head(max_rows)
    columns = [str(c) for c in pdf.columns]
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in pdf.itertuples(index=False):
        lines.append("| " + " | ".join(markdown_value(v) for v in row) + " |")
    return "\n".join(lines)


report_lines = [
    "# Resultados — Cuaderno ML 1: K-Means de ciclos de vida",
    "",
    f"- Generado en UTC: {datetime.now(timezone.utc).isoformat()}",
    f"- Gold: `{catalog}.{gold_schema}`",
    f"- Umbrales: min_months={MIN_MONTHS}, min_peak_players={MIN_PEAK}",
    f"- Juegos en el modelo: {n_model:,} (descartados por nulos: {n_population - n_model:,})",
    f"- k elegido: {K_FINAL} ({k_reason}) | silueta final: {SILHOUETTE_FINAL:.4f}",
    f"- Varianza explicada por PCA 2D: {explained[0]:.1%} + {explained[1]:.1%}",
    f"- Pares de variables con |r| > 0,85: {high_pairs if high_pairs else 'ninguno'}",
]
for title, pdf in REPORT:
    report_lines.extend(["", f"## {title}", "", to_markdown(pdf)])
report_text = "\n".join(report_lines) + "\n"

context_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
marker = "/src/ml"
if marker in context_path:
    repo_path = context_path.split(marker, 1)[0]
    workspace_repo_path = repo_path if repo_path.startswith("/Workspace") else "/Workspace" + repo_path
    output_dir = os.path.join(workspace_repo_path, "notebooks", "data_validations", "result_outputs")
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "resultados_ml_01_kmeans.md")
    with open(output_path + ".tmp", "w", encoding="utf-8", newline="\n") as handle:
        handle.write(report_text)
    os.replace(output_path + ".tmp", output_path)
    print(f"Markdown generado: {output_path}")
else:
    print(f"No se encontró '{marker}' en la ruta {context_path}; se imprime el informe:")
    print(report_text)